In [148]:
import pandas as pd

stud_info = pd.read_csv("tables/student_info.csv")
stud_info

,student_id,date_registered
0,258798,2022-01-01
1,258799,2022-01-01
2,258800,2022-01-01
3,258801,2022-01-01
4,258802,2022-01-01
...,...,...
37485,297050,2022-10-31
37486,297052,2022-10-31
37487,297053,2022-10-31
37488,297054,2022-10-31


In [149]:
subs = pd.read_csv("tables/student_purchases.csv")
subs

,purchase_id,student_id,purchase_type,date_purchased,date_refunded
0,15781,258800,2,2022-01-01,\N
1,15786,258803,2,2022-01-01,\N
2,15808,258862,2,2022-01-01,\N
3,15809,258865,2,2022-01-01,\N
4,15811,258878,2,2022-01-01,\N
...,...,...,...,...,...
3325,23345,296965,1,2022-10-31,\N
3326,23346,293600,2,2022-10-31,\N
3327,23347,296613,0,2022-10-31,\N
3328,23348,282158,1,2022-10-31,2022-11-03


In [150]:
subs.date_refunded[0]

'\\N'

In [151]:
import numpy as np
from dateutil.relativedelta import relativedelta

date_end = []
for i in range(len(subs)):
    if subs.purchase_type[i] == 0:
        n_months = 1
    elif subs.purchase_type[i] == 1:
        n_months = 3
    else:
        n_months = 12
    if subs.date_refunded[i] == "\\N":
        date_end.append(str(np.datetime64(subs.date_purchased[i]) + relativedelta(months=n_months)))
    else:
        date_end.append(subs.date_refunded[i])

date_end

['2023-01-01',
 '2023-01-01',
 '2023-01-01',
 '2023-01-01',
 '2023-01-01',
 '2022-02-01',
 '2022-01-06',
 '2023-01-02',
 '2023-01-02',
 '2022-02-07',
 '2022-02-02',
 '2023-01-03',
 '2023-01-03',
 '2023-01-03',
 '2022-01-31',
 '2023-01-04',
 '2023-01-04',
 '2022-02-04',
 '2022-02-04',
 '2022-02-04',
 '2022-01-24',
 '2022-02-04',
 '2022-02-05',
 '2022-02-05',
 '2022-01-24',
 '2022-02-05',
 '2023-01-06',
 '2023-01-06',
 '2022-02-06',
 '2022-01-14',
 '2023-01-07',
 '2023-01-07',
 '2022-02-07',
 '2023-01-07',
 '2023-01-08',
 '2022-02-08',
 '2023-01-08',
 '2023-01-08',
 '2023-01-08',
 '2023-01-08',
 '2022-02-09',
 '2022-02-09',
 '2022-02-09',
 '2022-02-09',
 '2022-02-09',
 '2023-01-09',
 '2022-02-09',
 '2022-02-10',
 '2023-01-10',
 '2023-01-10',
 '2022-02-10',
 '2023-01-10',
 '2022-02-10',
 '2022-02-10',
 '2023-01-10',
 '2023-01-10',
 '2023-01-10',
 '2022-02-10',
 '2023-01-11',
 '2023-01-11',
 '2023-01-11',
 '2022-02-11',
 '2023-01-11',
 '2023-01-12',
 '2023-01-12',
 '2022-02-11',
 '2023-01-

In [152]:
subs["date_end"] = date_end
subs

,purchase_id,student_id,purchase_type,date_purchased,date_refunded,date_end
0,15781,258800,2,2022-01-01,\N,2023-01-01
1,15786,258803,2,2022-01-01,\N,2023-01-01
2,15808,258862,2,2022-01-01,\N,2023-01-01
3,15809,258865,2,2022-01-01,\N,2023-01-01
4,15811,258878,2,2022-01-01,\N,2023-01-01
...,...,...,...,...,...,...
3325,23345,296965,1,2022-10-31,\N,2023-01-31
3326,23346,293600,2,2022-10-31,\N,2023-10-31
3327,23347,296613,0,2022-10-31,\N,2022-11-30
3328,23348,282158,1,2022-10-31,2022-11-03,2022-11-03


In [153]:
stud_info = stud_info[stud_info.student_id.isin(subs.student_id)]
stud_info

,student_id,date_registered
2,258800,2022-01-01
5,258803,2022-01-01
20,258818,2022-01-01
24,258822,2022-01-01
34,258832,2022-01-01
...,...,...
37273,296836,2022-10-30
37300,296863,2022-10-30
37302,296865,2022-10-30
37348,296911,2022-10-30


In [154]:
learning = pd.read_csv("tables/student_learning.csv")
learning.date_watched = pd.to_datetime(learning.date_watched)

total_mins = []
n_days = []
for i in range(len(stud_info)):
    sub_start, sub_end = (subs[subs.student_id == stud_info.student_id.iloc[i]].date_purchased.min(),
                          subs[subs.student_id == stud_info.student_id.iloc[i]].date_end.max())

    total_mins.append(learning[(learning.student_id == stud_info.student_id.iloc[i]) & (np.datetime64(sub_start) <= (learning.date_watched)) &
                               (learning.date_watched <= np.datetime64(sub_end))].minutes_watched.sum())
    
    purchases = subs[subs.student_id == stud_info.student_id.iloc[i]]
    n_days.append(sum([np.datetime64(e) - np.datetime64(s) for s, e in zip(purchases.date_purchased, purchases.date_end)]))

df = pd.DataFrame(data=[], columns=["student_id", "date_registered"])
df.student_id = stud_info.student_id
df.date_registered = stud_info.date_registered
df["total_minutes_watched"] = total_mins
df["num_paid_days"] = n_days
df

,student_id,date_registered,total_minutes_watched,num_paid_days
2,258800,2022-01-01,531.02,365 days
5,258803,2022-01-01,619.64,365 days
20,258818,2022-01-01,1392.10,365 days
24,258822,2022-01-01,382.31,31 days
34,258832,2022-01-01,227.53,31 days
...,...,...,...,...
37273,296836,2022-10-30,67.03,31 days
37300,296863,2022-10-30,81.58,31 days
37302,296865,2022-10-30,116.30,365 days
37348,296911,2022-10-30,20.72,26 days


In [159]:
def bins_generator(maximum):
    i = 0
    while i < maximum:
        yield i
        if i == 0:
            i = 30
        elif i < 480:
            i *= 2
        elif i == 480:
            i = 1000
        elif i < 4000:
            i += 1000
        elif i < 6000:
            i += 2000
        else:
            yield maximum
            i = maximum

pd.cut(df.total_minutes_watched, bins=bins_generator(df.total_minutes_watched.max()))

2         (480.0, 1000.0]
5         (480.0, 1000.0]
20       (1000.0, 2000.0]
24         (240.0, 480.0]
34         (120.0, 240.0]
               ...       
37273       (60.0, 120.0]
37300       (60.0, 120.0]
37302       (60.0, 120.0]
37348         (0.0, 30.0]
37400         (0.0, 30.0]
Name: total_minutes_watched, Length: 2338, dtype: category
Categories (11, interval[float64, right]): [(0.0, 30.0] < (30.0, 60.0] < (60.0, 120.0] < (120.0, 240.0] ... (2000.0, 3000.0] < (3000.0, 4000.0] < (4000.0, 6000.0] < (6000.0, 7698.46]]

In [160]:
df["buckets"] = pd.cut(df.total_minutes_watched, bins=bins_generator(df.total_minutes_watched.max()))
df

,student_id,date_registered,total_minutes_watched,num_paid_days,buckets
2,258800,2022-01-01,531.02,365 days,"(480.0, 1000.0]"
5,258803,2022-01-01,619.64,365 days,"(480.0, 1000.0]"
20,258818,2022-01-01,1392.10,365 days,"(1000.0, 2000.0]"
24,258822,2022-01-01,382.31,31 days,"(240.0, 480.0]"
34,258832,2022-01-01,227.53,31 days,"(120.0, 240.0]"
...,...,...,...,...,...
37273,296836,2022-10-30,67.03,31 days,"(60.0, 120.0]"
37300,296863,2022-10-30,81.58,31 days,"(60.0, 120.0]"
37302,296865,2022-10-30,116.30,365 days,"(60.0, 120.0]"
37348,296911,2022-10-30,20.72,26 days,"(0.0, 30.0]"


In [161]:
df.isna().sum()

student_id                 0
date_registered            0
total_minutes_watched      0
num_paid_days              0
buckets                  164
dtype: int64

In [166]:
df.buckets = df.buckets.cat.add_categories([0]).fillna(0)
df

,student_id,date_registered,total_minutes_watched,num_paid_days,buckets
2,258800,2022-01-01,531.02,365 days,"(480.0, 1000.0]"
5,258803,2022-01-01,619.64,365 days,"(480.0, 1000.0]"
20,258818,2022-01-01,1392.10,365 days,"(1000.0, 2000.0]"
24,258822,2022-01-01,382.31,31 days,"(240.0, 480.0]"
34,258832,2022-01-01,227.53,31 days,"(120.0, 240.0]"
...,...,...,...,...,...
37273,296836,2022-10-30,67.03,31 days,"(60.0, 120.0]"
37300,296863,2022-10-30,81.58,31 days,"(60.0, 120.0]"
37302,296865,2022-10-30,116.30,365 days,"(60.0, 120.0]"
37348,296911,2022-10-30,20.72,26 days,"(0.0, 30.0]"


In [167]:
df.isna().sum()

student_id               0
date_registered          0
total_minutes_watched    0
num_paid_days            0
buckets                  0
dtype: int64

In [168]:
df[df.buckets == 0]

,student_id,date_registered,total_minutes_watched,num_paid_days,buckets
2262,261084,2022-01-20,0.0,365 days,0
2394,261217,2022-01-22,0.0,365 days,0
2453,261276,2022-01-22,0.0,365 days,0
2467,261290,2022-01-23,0.0,19 days,0
2963,261789,2022-01-28,0.0,365 days,0
...,...,...,...,...,...
35204,294694,2022-10-14,0.0,365 days,0
35994,295506,2022-10-20,0.0,365 days,0
36037,295550,2022-10-21,0.0,365 days,0
36304,295823,2022-10-23,0.0,365 days,0


In [169]:
df

,student_id,date_registered,total_minutes_watched,num_paid_days,buckets
2,258800,2022-01-01,531.02,365 days,"(480.0, 1000.0]"
5,258803,2022-01-01,619.64,365 days,"(480.0, 1000.0]"
20,258818,2022-01-01,1392.10,365 days,"(1000.0, 2000.0]"
24,258822,2022-01-01,382.31,31 days,"(240.0, 480.0]"
34,258832,2022-01-01,227.53,31 days,"(120.0, 240.0]"
...,...,...,...,...,...
37273,296836,2022-10-30,67.03,31 days,"(60.0, 120.0]"
37300,296863,2022-10-30,81.58,31 days,"(60.0, 120.0]"
37302,296865,2022-10-30,116.30,365 days,"(60.0, 120.0]"
37348,296911,2022-10-30,20.72,26 days,"(0.0, 30.0]"


In [170]:
df.to_csv("data/sub_duration.csv", index=False)